# Vv–Gamma後処理・統計・描画

既存のVv–$w_{\mathrm{tip}}$ケースから `.stab` を収集し、定常旋回、ラダー限界旋回、ラダーステップ6自由度応答、横風突風6自由度応答を同じ条件で後処理する。その後、記事に掲載する範囲・成立ケース数・Pearson相関係数を再計算し、$V_v$–$\Gamma_{\mathrm{eff}}$ チャートを描く。


In [ ]:
from pathlib import Path
import math
import os
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.append(os.path.join("../../../"))
from src.VvGammaChart import (
    collect_vv_wtip_sweep_progress,
    plot_vv_gamma_contour_lines,
    plot_vv_gamma_contour_panel,
    postprocess_vv_gamma_cases,
    read_reference_wing_summary,
)

plt.rcParams["mathtext.fontset"] = "stix"


## 1. ケース一覧と等価上反角の半スパン

ケースディレクトリ名から実際に存在する $V_v$ と翼端たわみ量を取得する。$\Gamma_{\mathrm{eff}}$ の計算に使う半スパンは固定値で与えず、基準OpenVSPモデルの参照翼幅から取得する。


In [ ]:
base_vsp3_path = Path("../../models/SampleGlider/SampleGlider.0G.vsp3")
output_dir = Path(".")

case_pattern = re.compile(r"^vv_(m?\d+p\d{5})_wtip_(m?\d+p\d{5})$")
cases = []
for path in output_dir.iterdir():
    if not path.is_dir():
        continue
    match = case_pattern.fullmatch(path.name)
    if match is None:
        continue
    vv = float(match.group(1).replace("m", "-").replace("p", "."))
    tip_deflection = float(match.group(2).replace("m", "-").replace("p", "."))
    cases.append((vv, tip_deflection))

if not cases:
    raise FileNotFoundError(f"Vv/wtip case directories were not found in {output_dir.resolve()}.")

vv_values = np.array(sorted({vv for vv, _ in cases}), dtype=float)
tip_deflections = np.array(sorted({wtip for _, wtip in cases}), dtype=float)

reference_wing = read_reference_wing_summary(base_vsp3_path)
gamma_semispan = float(reference_wing["span"]) / 2.0

print(f"case count       = {len(vv_values) * len(tip_deflections)}")
print(f"reference span   = {reference_wing['span']}")
print(f"gamma semispan   = {gamma_semispan}")
print("vv_values        =", vv_values)
print("tip_deflections  =", tip_deflections)


In [ ]:
sweep = collect_vv_wtip_sweep_progress(
    vv_values,
    tip_deflections,
    output_dir,
    # gamma_semispan=gamma_semispan,
    base_vsp3_path=base_vsp3_path,
    include_incomplete_cases=False,
    output_csv_path=output_dir / "vv_wtip_stability_sweep_progress.csv",
)

print(f"completed .stab cases = {len(sweep)}")
sweep.head()


## 2. 後処理条件

記事掲載値を生成する条件をここへ集約する。角度は内部ではrad、最終CSVでは通常角度をdeg、通常角速度をdeg/sへ変換する。


In [ ]:
input_csv = output_dir / "vv_wtip_stability_sweep_progress.csv"
output_csv = output_dir / "vv_gamma_chart.csv"
history_output_dir = output_dir / "6dof_histories"

mass = 100.0
inertia = {
    "Ixx": 1000.0,
    "Iyy": 75.0,
    "Izz": 1000.0,
    "Ixz": 0.0,
}

# ラダーステップ6自由度応答
delta_r = math.radians(-10.0)
target_delta_phi = math.radians(+2.0)
t_final = 99.0
max_step = 0.01

delta_a = 0.0
delta_e = None
trim_elevator = True
theta_hold = True
theta_hold_kp = 0.3
theta_hold_kq = 0.8
trim_thrust = True

# 固定バンク角の滑空旋回トリム
turn_trim_mode = "gliding"
turn_trim_phi = math.radians(+2.0)
turn_trim_delta_a = 0.0
turn_trim_bounds = (
    {
        "alpha": math.radians(-4.0),
        "theta": math.radians(-10.0),
        "delta_e": math.radians(-20.0),
        "delta_r": math.radians(-30.0),
    },
    {
        "alpha": math.radians(+4.0),
        "theta": math.radians(+10.0),
        "delta_e": math.radians(+20.0),
        "delta_r": math.radians(+30.0),
    },
)

# 正負のラダー限界における滑空旋回トリム
rudder_limit_turn_mode = "gliding"
rudder_limit_turn_delta_r_max = math.radians(10.0)
rudder_limit_turn_delta_a = 0.0
rudder_limit_turn_bounds = (
    {
        "alpha": math.radians(-4.0),
        "beta": math.radians(-20.0),
        "phi": math.radians(-60.0),
        "theta": math.radians(-10.0),
        "delta_e": math.radians(-10.0),
    },
    {
        "alpha": math.radians(+4.0),
        "beta": math.radians(+20.0),
        "phi": math.radians(+60.0),
        "theta": math.radians(+10.0),
        "delta_e": math.radians(+10.0),
    },
)

# 1-cosine横風突風6自由度応答
crosswind_gust_Uds = 3.0
crosswind_gust_H = 30.0 * 0.3048
crosswind_gust_t_final = 3.0
crosswind_gust_start_time = 0.0
crosswind_gust_delta_a = 0.0
crosswind_gust_delta_e = None
crosswind_gust_delta_r = 0.0

print("input_csv :", input_csv)
print("output_csv:", output_csv)
print("history   :", history_output_dir)


## 3. 一括後処理

231ケースの一括評価では、時刻歴CSVと図をケースごとに保存すると時間と容量を大きく消費するため、既定では保存しない。代表ケースだけを確認するときに各フラグを `True` にする。


In [ ]:
# df = postprocess_vv_gamma_cases(
#     input_csv,
#     stab_path_column="stab_path",
#     # gamma_semispan_column="gamma_semispan",
#     # gamma_n_span=1001,
#     control_map=None,
#     mass=mass,
#     inertia=inertia,
#     rho=None,
#     g=9.80665,
#     run_6dof=True,
#     delta_r=delta_r,
#     target_delta_phi=target_delta_phi,
#     stop_6dof_at_target_delta_phi=True,
#     t_final=t_final,
#     delta_a=delta_a,
#     delta_e=delta_e,
#     trim_elevator=trim_elevator,
#     theta_hold=theta_hold,
#     theta_ref=None,
#     theta_hold_kp=theta_hold_kp,
#     theta_hold_kq=theta_hold_kq,
#     delta_e_min=math.radians(-20.0),
#     delta_e_max=math.radians(+20.0),
#     thrust=None,
#     trim_thrust=trim_thrust,
#     phi0=0.0,
#     theta0=None,
#     psi0=0.0,
#     max_step=max_step,
#     rtol=1.0e-8,
#     atol=1.0e-10,
#     turn_trim_mode=turn_trim_mode,
#     turn_trim_phi=turn_trim_phi,
#     turn_trim_delta_a=turn_trim_delta_a,
#     turn_trim_initial_guess=None,
#     turn_trim_bounds=turn_trim_bounds,
#     turn_trim_residual_tol=1.0e-6,
#     rudder_limit_turn_mode=rudder_limit_turn_mode,
#     rudder_limit_turn_delta_r_max=rudder_limit_turn_delta_r_max,
#     rudder_limit_turn_delta_a=rudder_limit_turn_delta_a,
#     rudder_limit_turn_initial_guess=None,
#     rudder_limit_turn_bounds=rudder_limit_turn_bounds,
#     rudder_limit_turn_residual_tol=1.0e-6,
#     write_6dof_history=False,
#     plot_6dof_history=False,
#     run_crosswind_gust_6dof=True,
#     crosswind_gust_Uds=crosswind_gust_Uds,
#     crosswind_gust_H=crosswind_gust_H,
#     crosswind_gust_t_final=crosswind_gust_t_final,
#     crosswind_gust_start_time=crosswind_gust_start_time,
#     crosswind_gust_delta_a=crosswind_gust_delta_a,
#     crosswind_gust_delta_e=crosswind_gust_delta_e,
#     crosswind_gust_delta_r=crosswind_gust_delta_r,
#     write_crosswind_gust_history=False,
#     plot_crosswind_gust_history=False,
#     history_output_dir=history_output_dir,
#     output_csv_path=output_csv,
#     verbose=2,
# )

# df.head()


## 4. 記事掲載値の再計算

成立ケース数、各指標の範囲、簡易式とのPearson相関係数をCSVから再計算する。記事を更新するときは、ここで表示・保存される3表を根拠にする。


In [ ]:
df = pd.read_csv(output_csv)

bool_columns = [
    "postprocess_passed",
    "turn_trim_passed",
    "rudder_limit_turn_passed",
    "sixdof_roll_response_reached",
    "crosswind_gust_success",
]
bool_masks = {
    column: df[column].astype(str).str.lower().isin(["true", "1", "yes"])
    for column in bool_columns
    if column in df.columns
}

case_counts = pd.DataFrame(
    [
        {"item": "all rows", "count": len(df)},
        {"item": "postprocess passed", "count": int(bool_masks["postprocess_passed"].sum())},
        {"item": "fixed-bank turn trim passed", "count": int(bool_masks["turn_trim_passed"].sum())},
        {"item": "rudder-limit turn passed", "count": int(bool_masks["rudder_limit_turn_passed"].sum())},
        {"item": "6DOF target bank reached", "count": int(bool_masks["sixdof_roll_response_reached"].sum())},
        {"item": "crosswind-gust 6DOF passed", "count": int(bool_masks["crosswind_gust_success"].sum())},
    ]
)

rudder_column = df.loc[df["control_column_delta_r"].notna(), "control_column_delta_r"].iloc[0]
range_columns = [
    "CY_Beta", "CMl_Beta", "CMn_Beta",
    "CY_p", "CMl_p", "CMn_p",
    "CY_r", "CMl_r", "CMn_r",
    f"CY_{rudder_column}", f"CMl_{rudder_column}", f"CMn_{rudder_column}",
    # "CD_Base",
]
metric_ranges = pd.DataFrame(
    [
        {
            "column": column,
            "count": int(pd.to_numeric(df[column], errors="coerce").notna().sum()),
            "min": pd.to_numeric(df[column], errors="coerce").min(),
            "median": pd.to_numeric(df[column], errors="coerce").median(),
            "max": pd.to_numeric(df[column], errors="coerce").max(),
        }
        for column in range_columns
        if column in df.columns
    ]
)

csv_output_dir = output_dir / "vv_gamma_chart_csv"
csv_output_dir.mkdir(parents=True, exist_ok=True)
case_counts.to_csv(csv_output_dir / "vv_gamma_case_counts.csv", index=False)
metric_ranges.to_csv(csv_output_dir / "vv_gamma_metric_ranges.csv", index=False)

display(case_counts)
display(metric_ranges)


## 5. 描画設定


In [ ]:
VV_TICKS = np.linspace(0.001, 0.006, 6)
GAMMA_TICKS = np.linspace(0.0, 9.0, 10)
GRID_SIZE = 256
LEVELS = 256
METHOD = "linear"
FALLBACK_METHOD = "nearest"
SHOW_POINTS = False

COLORBAR_TICKS_BY_COLUMN = {
    "simple_rudder_limit_turn_max_abs_phi": np.linspace(0.0, 7.0, 8),
    "rudder_limit_turn_max_abs_phi": np.linspace(0.0, 7.0, 8),
    "turn_trim_delta_r": np.linspace(-10.0, 10.0, 11),
    "simple_rudder_turn_K_phi": np.linspace(0.0, 0.62264444, 8),
    "turn_trim_beta": np.linspace(0.0, 15, 8),
}

PANEL_KWARGS = {
    "grid_size": GRID_SIZE,
    "levels": LEVELS,
    "method": METHOD,
    "fallback_method": FALLBACK_METHOD,
    "colorbar_ticks_by_column": COLORBAR_TICKS_BY_COLUMN,
    "x_ticks": VV_TICKS,
    "y_ticks": GAMMA_TICKS,
    "show_points": SHOW_POINTS,
    "show_colorbar": True,
}

plot_output_dir = output_dir / "vv_gamma_chart_plots"
plot_output_dir.mkdir(parents=True, exist_ok=True)

def save_figure(fig, stem):
    fig.tight_layout()
    fig.savefig(plot_output_dir / f"{stem}.png", dpi=200, bbox_inches="tight")
    # fig.savefig(plot_output_dir / f"{stem}.pdf", bbox_inches="tight")


## 6. 横・方向の空力微係数

列は左から $C_Y,C_l,C_n$、行は上から $\beta,\hat p,\hat r,\delta_a,\delta_r$ とする。Control Group列は後処理結果に保存された対応をそのまま使い、Notebook内で再解決しない。


In [ ]:
delta_a_column = df.loc[df["control_column_delta_a"].notna(), "control_column_delta_a"].iloc[0]
delta_r_column = df.loc[df["control_column_delta_r"].notna(), "control_column_delta_r"].iloc[0]

lateral_columns = [("CY", r"C_Y"), ("CMl", r"C_l"), ("CMn", r"C_n")]
derivative_rows = [
    ("Beta", r"\beta"),
    ("p", r"\hat{p}"),
    ("r", r"\hat{r}"),
    # (delta_a_column, r"\delta_a"),
    (delta_r_column, r"\delta_r"),
]

fig, axes = plt.subplots(
    len(derivative_rows),
    len(lateral_columns),
    figsize=(5*len(lateral_columns), 3*len(derivative_rows)+1),
    sharex=True,
    sharey=True,
)
axes = axes.reshape(len(derivative_rows), len(lateral_columns),)
for row_index, (suffix, variable_label) in enumerate(derivative_rows):
    for column_index, (prefix, coefficient_label) in enumerate(lateral_columns):
        column = f"{prefix}_{suffix}"
        label = rf"$\partial {coefficient_label}/\partial {variable_label}$"
        plot_vv_gamma_contour_panel(
            df,
            column,
            ax=axes[row_index, column_index],
            label=label,
            **PANEL_KWARGS,
        )

save_figure(fig, "turn-manuver_017_vv_gamma_lateral_derivatives")
plt.show()


## 7. ラダーのみ旋回の評価指標


In [ ]:
panel_groups = {
    "vv_gamma_spiral_stability": [
        ("spiral_margin", None),
        ("turn_trim_delta_r", None),
    ],
    "vv_gamma_rudder_turn_indices": [
        ("rudder_limit_turn_max_abs_phi", None),
        # ("simple_rudder_limit_turn_max_abs_phi", None),
        ("simple_rudder_turn_K_phi", None),
    ],
    "vv_gamma_rudder_roll_response": [
        ("sixdof_roll_response_phi_rate_per_delta_r", None),
        ("simple_rudder_roll_index", None),
    ],
    "vv_gamma_gust_response": [
        ("crosswind_gust_max_phi_delta", None),
        ("crosswind_gust_roll_index", None),
    ],
    "vv_gamma_drag": [
        ("CD_Base", None),
        ("turn_trim_beta", None),
    ],
}

correlation_rows = []
for stem, panels in panel_groups.items():
    (x_column, _), (y_column, _) = panels

    # correlation
    pair = df[[x_column, y_column]].apply(pd.to_numeric, errors="coerce").dropna()
    correlation_rows.append(
        {
            "comparison": label,
            "x_column": x_column,
            "y_column": y_column,
            "count": len(pair),
            "pearson_r": pair[x_column].corr(pair[y_column], method="pearson"),
        }
    )
    print(f"r({x_column}, {y_column}) = {correlation_rows[-1]['pearson_r']}")

    for column, _ in panels:
        min = pd.to_numeric(df[column], errors="coerce").min()
        median = pd.to_numeric(df[column], errors="coerce").median()
        max =  pd.to_numeric(df[column], errors="coerce").max()
        print(f"{min:10.6f} < {column} < {max:10.6f}")

    fig, axes = plt.subplots(1, len(panels), figsize=(6 * len(panels), 4.5), squeeze=False)
    for ax, (column, label) in zip(axes.flat, panels):
        plot_vv_gamma_contour_panel(df, column, ax=ax, label=label, **PANEL_KWARGS)
    save_figure(fig, stem)
    plt.show()

correlations = pd.DataFrame(correlation_rows)
correlations.to_csv(csv_output_dir / "vv_gamma_correlations.csv", index=False)
display(correlations)


## 8. 設計指標の重ね描画


In [ ]:
# グラフ全体のデフォルトフォントサイズを変更（デフォルトは10〜12）
plt.rcParams["font.size"] = 12

In [ ]:
CONTOUR_LINE_SETTINGS = {
    "spiral_margin": {
        "levels": [0.0],
        "color": "k",
        "linestyle": "-",
        "linewidth_min": 2.0,
        "linewidth_max": 2.0,
        "linewidth_scale": "value",
        "show_labels": True,
        "method": "cubic",
    },
    "rudder_limit_turn_max_abs_phi": {
        "levels": np.linspace(0.0, 7.0, 8),
        "color": "tab:blue",
        "linestyle": "-",
        "linewidth_min": 0.5,
        "linewidth_max": 1.0,
        "linewidth_scale": "value",
        "show_labels": True,
        "method": "cubic",
    },
    "sixdof_roll_response_phi_rate_per_delta_r": {
        "levels": np.linspace(-0.07, 0.0, 15),
        "color": "tab:orange",
        "linestyle": "-",
        "linewidth_min": 0.5,
        "linewidth_max": 1.0,
        "linewidth_scale": "absolute",
        "show_labels": True,
        "label_format": "%.3f",
        "method": "cubic",
    },
    "crosswind_gust_max_phi_delta": {
        "levels": np.linspace(-3.0, 1.0, 9),
        "color": "tab:green",
        "linestyle": "-",
        "linewidth_min": 0.5,
        "linewidth_max": 1.0,
        "linewidth_scale": "absolute-",
        "show_labels": True,
        "method": "cubic",
    },
    "CD_Base": {
        "levels": np.linspace(0.0213, 0.0217, 9),
        "color": "tab:orange",
        "linestyle": "-",
        "linewidth_min": 0.5,
        "linewidth_max": 1.0,
        "linewidth_scale": "absolute",
        "show_labels": True,
        "label_format": "%.4f",
        "method": "cubic",
    },
}

fig, ax = plt.subplots(figsize=(8, 7))
plot_vv_gamma_contour_lines(
    df,
    CONTOUR_LINE_SETTINGS,
    ax=ax,
    grid_size=GRID_SIZE,
    method=METHOD,
    fallback_method=FALLBACK_METHOD,
    x_ticks=VV_TICKS,
    y_ticks=GAMMA_TICKS,
    show_points=SHOW_POINTS,
    title="Vv-Gamma Chart",
)
save_figure(fig, "vv_gamma_contour_line_overlay")
plt.show()


## 9. 出力ファイル


In [ ]:
for path in sorted(output_dir.glob("vv_gamma_*")):
    print(path)
for path in sorted(plot_output_dir.glob("vv_gamma_*")):
    print(path)
